In [18]:
import pandas as pd
import numpy as np
from tensorflow.keras.layers import Dense, Dropout, Flatten,LSTM,GRU
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential

In [19]:
clns=["unit_number","time_cycles","op_setting_1","op_setting_2","op_setting_3"]+[f"sensor{i}" for i in range(1,22)]
fe=["op_setting_1","op_setting_2","op_setting_3"]+[f"sensor{i}" for i in range(1, 22)]

print(len(clns))

26


In [20]:
train_1=pd.read_csv("./nasa/train_FD001.txt",sep=r"\s+",header=None,names=clns)

In [21]:
test_1=pd.read_csv("./nasa/test_FD001.txt",sep=r"\s+",header=None,names=clns)

In [22]:
rul_1=pd.read_csv("./nasa/rul_FD001.txt",sep=r"\s+",header=None,names=["rul"])

In [23]:
mx_c=train_1.groupby("unit_number")["time_cycles"].transform("max")
train_1["rul"]=mx_c-train_1["time_cycles"]

In [24]:
sc=MinMaxScaler()
train_1[fe]=sc.fit_transform(train_1[fe])
test_1[fe]=sc.transform(test_1[fe])

In [25]:
s_l=30
xl=[]
yl=[]
for i in train_1["unit_number"].unique():
    en_data=train_1[train_1["unit_number"]==i].sort_values("time_cycles")
    data=en_data[fe].values
    rul_val=en_data["rul"].values

    for j in range(0,len(data)-s_l+1):
        wi=data[j:j+s_l]
        tar=rul_val[j+s_l-1]
        xl.append(wi)
        yl.append(tar)
x_train=np.array(xl)
y_train=np.array(yl)

In [26]:
print(len(fe))

24


In [27]:
model=Sequential()
model.add(GRU(64,return_sequences=True,input_shape=(s_l,24)))
model.add(GRU(64,return_sequences=True))
model.add(Dropout(0.2))
model.add(GRU(32,return_sequences=True))
model.add(Dropout(0.2))
model.add(GRU(16,return_sequences=False))
model.add(Dense(1))

C:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [28]:
compile=model.compile(optimizer='adam',loss='mse',metrics=['mae'])

In [29]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_4 (GRU)                     │ (None, 30, 64)         │        17,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ (None, 30, 64)         │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_6 (GRU)                     │ (None, 30, 32)         │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 30, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 16)             │         2,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,065 (211.19 KB)

 Trainable params: 54,065 (211.19 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
history=model.fit(x_train,y_train,validation_split=0.2,epochs=100,batch_size=32)

Epoch 1/40
444/444 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - loss: 10217.8027 - mae: 82.7561 - val_loss: 13908.4668 - val_mae: 94.2916
Epoch 2/40
444/444 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - loss: 9032.8857 - mae: 76.3541 - val_loss: 12626.4971 - val_mae: 88.4448
Epoch 3/40
444/444 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - loss: 8041.6113 - mae: 71.0317 - val_loss: 11500.9619 - val_mae: 83.3884
Epoch 4/40
444/444 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - loss: 7182.5571 - mae: 66.4435 - val_loss: 10507.2920 - val_mae: 78.9948
Epoch 5/40
444/444 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - loss: 6441.4087 - mae: 62.5003 - val_loss: 9632.7578 - val_mae: 75.1823
Epoch 6/40
444/444 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - loss: 5807.7900 - mae: 59.1655 - val_loss: 8868.4531 - val_mae: 71.9031
Epoch 7/40
444/444 ━━━━━━━━━━━━━━━━━━━━ 15s 34ms/step - loss: 5271.1724 - mae: 56.4059 - val_loss: 8204.0479 - val_mae: 69.1053
Epoch 8/40
444/444 ━━━━━━━━━━━━━━━━━━━━ 19s 43ms/step - loss: 4821.7705 - mae: 54.1143 - val_loss: 763